# food-fraud-cv — Pipeline real en Colab (GPU)

Corre el pipeline **real** end-to-end sobre GPU:
imágenes reales (HF `Densu341/Fresh-rotten-fruit`) + **difusión real** (SD 1.5 inpainting) para fabricar los `fake-damaged` + **fine-tune** de ResNet/ViT + evaluación cost-sensitive.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU` (una T4 gratis alcanza).

Datos 100% públicos · cero datos de Rappi.

In [ ]:
# 1) Verificar GPU
!nvidia-smi -L || echo 'SIN GPU — andá a Entorno de ejecución → Cambiar tipo de entorno → GPU'

In [ ]:
# 2) Clonar el repo (público)
import os
if not os.path.isdir('food-fraud-cv'):
    !git clone -q https://github.com/nicoespa/food-fraud-cv.git
else:
    !cd food-fraud-cv && git pull -q
%cd food-fraud-cv

In [ ]:
# 3) Dependencias. Colab ya trae torch+CUDA, numpy, pandas, sklearn, matplotlib.
#    Instalamos solo el stack de difusión + datos (no reinstalar torch para no romper CUDA).
!pip install -q diffusers timm datasets accelerate

In [ ]:
# 4) PRUEBA CHICA primero (valida que datos→difusión→fine-tune corre; ~5 min en GPU).
#    n chico + pocos pasos de difusión. Si esto anda, corré la celda 5 (la informativa).
!python scripts/run_real.py --config configs/real.yaml --n-per-group 30 --steps 8

In [ ]:
# 5) CORRIDA COMPLETA INFORMATIVA (escala real; ~20-30 min en T4).
#    Usa configs/real.yaml: n_per_group=150, steps=25 (fakes sutiles), backbone resnet50.
#    Más imágenes + más pasos de difusión = el resultado SÍ discrimina entre modelos.
!python scripts/run_real.py --config configs/real.yaml

In [ ]:
# 6) Resultados
import json, pandas as pd
print(json.dumps(json.load(open('results/experiment_real.json'))['cost_improvement_D0_to_D2'], indent=2))
pd.read_csv('results/experiment_real.csv')

## Guardar resultados (opcional)
El disco de Colab es efímero. Para conservar el corpus generado + resultados, montá Drive o subí a HF Hub.

In [ ]:
# Guardar resultados en Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/food-fraud-cv' && cp results/experiment_real.* '/content/drive/MyDrive/food-fraud-cv/'
print('Resultados copiados a Drive/food-fraud-cv')